<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

| Country        | States                                                       | Producing Regions                                                                        | Productivity Data | Soil File | Average Cycle |
|----------------|--------------------------------------------------------------|------------------------------------------------------------------------------------------|-------------------|-----------|---------------|
| United States  | Iowa, Illinois, Nebraska, Minnesota, Indiana                 | Corn Belt (IA, IL, IN); East/Center of NE; South of MN                                   | USDA              | EC6       | Apr – Nov     |
| China          | Heilongjiang, Jilin, Nei Mongol, Shandong, Henan             | Northeast and North China Plains                                                         | NBS               | EC6       | Apr – Oct     |
| Brazil         | Mato Grosso, Paraná, Goiás, Mato Grosso do Sul, Minas Gerais | MT (Mid-North), PR (West), GO (South), MS (Southwest), MG (Triangle)                     | SIDRA-IBGE        | EC3       | Jan – Sep     |
| European Union | France, Romania, Poland, Hungary, Italy                      | FRA (N. Aquitaine), ROM (South), POL (Lower Silesia), HUN (Great Plain), ITA (Po Valley) | AGRI4CAST         | EC2       | Mar – Dec     |
| Argentina      | Córdoba, Buenos Aires, Santa Fé, Santiago del Estero         | Core Zone (North BA, South SF, Center CD); Southeast SDE                                 | BC EXPLORER       | EC6       | Sep – Aug     |
| India          | Karnataka, Madhya Pradesh, Bihar, Tamil Nadu, Telangana      | Ballari-KA, Chhindwara-MP, "Corn Zone"-BI                                                | DES               | EC4       | Mar – Dec     |
| Mexico         | Sinaloa, Jalisco, Michoacán, Guerrero, Chiapas               | Sinaloa Valleys; Ciénega/Altos Regions (Jalisco)                                         | DGSIAP            | EC4       | Apr – Feb     |

Soil File Legend:

* EC2: medium texture soils
* EC3: medium-fine soils
* EC4: fine soils
* EC6: fine and permeable soils <br>
The average cycle comprises the period from sowing to harvest.

## Baixando dados brutos

ERA5

### 📊 Resumo de Metadados: Variáveis ARCO-ERA5

| Grandeza                   | Nome Longo                        | Nome Curto | Unidade | Shape (Time, Lat, Lon) |
|:---------------------------|:----------------------------------|:-----------|:--------|:-----------------------|
| **Temperature (2m)**       | 2 metre temperature               | `t2m`      | K       | [1323648, 721, 1440]   |
| **Dewpoint Temp. (2m)**    | 2 metre dewpoint temperature      | `d2m`      | K       | [1323648, 721, 1440]   |
| **U Wind Component (10m)** | 10 metre U wind component         | `u10`      | m s**-1 | [1323648, 721, 1440]   |
| **V Wind Component (10m)** | 10 metre V wind component         | `v10`      | m s**-1 | [1323648, 721, 1440]   |
| **Total Precipitation**    | Total precipitation               | `tp`       | m       | [1323648, 721, 1440]   |
| **Solar Radiation**        | Surface solar radiation downwards | `ssrd`     | J m**-2 | [1323648, 721, 1440]   |

**Coordenadas Globais:**
*   **Latitude:** -90.00° a 90.00° (Ordem Decrescente)
*   **Longitude:** 0.00° a 359.75°

In [ ]:
import gc
import logging

import os

import dask
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# 2. DEFINIÇÃO DE PARÂMETROS E DIRETÓRIOS
dask.config.set(scheduler='single-threaded')
logging.basicConfig(level=logging.INFO, format='%(message)s')

ZARR_URL = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
VARIABLES_TO_LOAD = [
    "2m_temperature", "2m_dewpoint_temperature", "10m_u_component_of_wind",
    "10m_v_component_of_wind", "total_precipitation", "surface_solar_radiation_downwards"
]

ANOS = range(1940, 2026)
VARIAVEIS_ALVO = ['tasmax', 'tasmin', 'hurs', 'sfcWind', 'rsds', 'pr']

# Estrutura Robusta e Atomizada no Drive
DIRETORIO_BASE = os.getcwd()
TEMP_DIR = os.path.join(DIRETORIO_BASE, 'temp_era5')

# Criação das subpastas por variável para organização do Data Lake
for var in VARIAVEIS_ALVO:
    os.makedirs(os.path.join(DIRETORIO_BASE, var), exist_ok=True)

# 3. FUNÇÕES DE PROCESSAMENTO (com try/except e limpeza)
def limpar_temp():
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR, ignore_errors=True)
    os.makedirs(TEMP_DIR, exist_ok=True)

    for dask_trash in glob.glob('/tmp/dask-worker-space*'):
        try: shutil.rmtree(dask_trash, ignore_errors=True)
        except: pass

def wind10to2(wind10):
    fator = np.log10(2.0 / 0.033) / np.log10(10.0 / 0.033)
    return wind10 * fator

def formatar_dataset_saida(da, var_name, units):
    """Encapsula o DataArray em um Dataset com metadados e coordenadas padronizadas."""
    ds_out = xr.Dataset({var_name: da.astype('float32')})
    ds_out[var_name].attrs['units'] = units

    # Alinhamento espacial (NEX-GDDP Padrão)
    ds_out.coords['longitude'] = (ds_out.coords['longitude'] + 180) % 360 - 180
    ds_out = ds_out.sortby(ds_out.longitude)
    ds_out = ds_out.sortby(ds_out.latitude)
    ds_out = ds_out.rename({'latitude': 'lat', 'longitude': 'lon'})

    ds_out[var_name].encoding.clear()
    return ds_out

# 4. LOOP PRINCIPAL (com lógica de Checkpoint/Skip)
try:
    logging.info("Mapeando ARCO-ERA5 via GCS (Lazy Load)...")
    ds_global = xr.open_zarr(
        ZARR_URL,
        consolidated=True,
        storage_options={'token': 'anon'},
        chunks={'time': 24}
    )[VARIABLES_TO_LOAD]

    for ano in ANOS:
        logging.info(f"\n[{ano}] Inicializando recortes anuais...")
        ds_ano = ds_global.sel(time=slice(f'{ano}-01-01', f'{ano}-12-31'))

        if len(ds_ano.time) == 0:
            continue

        # Agregações base (Lazy) para o ano, evitam recomputação do grafo S3
        daily_max = ds_ano['2m_temperature'].resample(time='1D').max(skipna=False)
        daily_min = ds_ano['2m_temperature'].resample(time='1D').min(skipna=False)
        daily_mean = ds_ano[['2m_dewpoint_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind']].resample(time='1D').mean(skipna=False)
        daily_sum = ds_ano[['total_precipitation', 'surface_solar_radiation_downwards']].resample(time='1D').sum(skipna=False)

        # Loop de Variáveis (Isolamento total de I/O e RAM)
        for var_alvo in VARIAVEIS_ALVO:
            arquivo_saida = os.path.join(DIRETORIO_BASE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")

            if os.path.exists(arquivo_saida) and os.path.getsize(arquivo_saida) > 10240:
                logging.info(f"  -> [{var_alvo}] Checkpoint detectado. Pulando.")
                continue

            logging.info(f"  -> [{var_alvo}] Processando e gravando...")
            limpar_temp()
            ds_out = None

            try:
                if var_alvo == 'tasmax':
                    da = daily_max - 273.15
                    ds_out = formatar_dataset_saida(da, var_alvo, 'degC')

                elif var_alvo == 'tasmin':
                    da = daily_min - 273.15
                    ds_out = formatar_dataset_saida(da, var_alvo, 'degC')

                elif var_alvo == 'hurs':
                    tasmax_k = daily_max
                    tasmin_k = daily_min
                    Tdew_c = daily_mean['2m_dewpoint_temperature'] - 273.15

                    Ea = 0.6108 * np.exp((17.27 * Tdew_c) / (Tdew_c + 237.3))
                    Es_tmax = 0.6108 * np.exp((17.27 * (tasmax_k - 273.15)) / ((tasmax_k - 273.15) + 237.3))
                    Es_tmin = 0.6108 * np.exp((17.27 * (tasmin_k - 273.15)) / ((tasmin_k - 273.15) + 237.3))
                    Es = (Es_tmax + Es_tmin) / 2.0

                    da = 100 * (Ea / Es)
                    da = da.clip(min=0, max=100)
                    ds_out = formatar_dataset_saida(da, var_alvo, '%')

                elif var_alvo == 'sfcWind':
                    u = daily_mean['10m_u_component_of_wind']
                    v = daily_mean['10m_v_component_of_wind']
                    wind10 = np.sqrt(u**2 + v**2)
                    da = wind10to2(wind10)
                    da = da.clip(min=0)
                    ds_out = formatar_dataset_saida(da, var_alvo, 'm s-1')

                elif var_alvo == 'rsds':
                    da = daily_sum['surface_solar_radiation_downwards'] / 1e6
                    da = da.clip(min=0)
                    ds_out = formatar_dataset_saida(da, var_alvo, 'MJ m-2 day-1')

                elif var_alvo == 'pr':
                    da = daily_sum['total_precipitation'] * 1000.0
                    da = da.clip(min=0)
                    ds_out = formatar_dataset_saida(da, var_alvo, 'mm/day')

                # Escrita isolada
                with ProgressBar():
                    ds_out.to_netcdf(arquivo_saida, engine='h5netcdf', format='NETCDF4')

            except Exception as e:
                logging.error(f"  ❌ Falha no processamento de {var_alvo} ({ano}): {e}")
                if os.path.exists(arquivo_saida):
                    os.remove(arquivo_saida)
            finally:
                if 'da' in locals(): del da
                if 'ds_out' in locals(): del ds_out
                gc.collect()

        # Limpeza do ano
        del ds_ano, daily_max, daily_min, daily_mean, daily_sum
        gc.collect()

except KeyboardInterrupt:
    logging.info("\n⛔ Interrupção manual. Checkpoints por variável preservados.")
except Exception as e:
    logging.error(f"❌ Erro crítico GCS: {e}")
finally:
    if 'ds_global' in locals(): del ds_global
    limpar_temp()
    gc.collect()

# 5. CONSOLIDAÇÃO DOS DADOS (Opcional)
# Arquitetura Omitida: Como estabelecido, o arquivo monolítico final não será gerado.
# Para carregar virtualmente uma variável específica (ex: tasmax do ERA5) em etapas futuras de Machine Learning:
# ds_virtual = xr.open_mfdataset('/content/drive/Shareddrives/GAS-Henrique/ERA5_Atomizado/tasmax/ERA5_tasmax_*.nc4', engine='h5netcdf')

2. CMIP6 Data Acquisition (Global Climate Models)

To acquire CMIP6 data, we use the `intake_esgf` library. This tool allows us to query the Earth System Grid Federation (ESGF) nodes directly and download the NetCDF files programmatically.

**Applied Search Filters:**
* **source_id**: Selected climate models (`CMCC-ESM2`, `MPI-ESM1-2-HR`, `MRI-ESM2-0`, `NorESM2-MM`).
* **experiment_id**: Reference scenario (`historical`) and future projection scenarios (`ssp126`, `ssp245`, `ssp585`).
* **variable_id**: Our variables of interest (temperature, relative humidity, wind speed, precipitation, and solar radiation).
* **frequency / table_id**: Data with daily temporal resolution (`day`).
* **variant_label**: We selected the `r1i1p1f1` ensemble (first realization, initialization, physics, and forcing) to standardize the datasets across different models.
* **latest**: Setting this to `True` ensures we download the most recently updated and published versions on the ESGF node.

### 📊 Resumo de Metadados: NEX-GDDP-CMIP6 (GFDL-ESM4)

| Grandeza              | Nome Longo                                 | Nome Curto | Unidade    | Shape (Time, Lat, Lon) |
|:----------------------|:-------------------------------------------|:-----------|:-----------|:-----------------------|
| **Relative Humidity** | Near-Surface Relative Humidity             | `hurs`     | %          | [365, 600, 1440]       |
| **Precipitation**     | Precipitation                              | `pr`       | kg m-2 s-1 | [365, 600, 1440]       |
| **Solar Radiation**   | Surface Downwelling Shortwave Radiation    | `rsds`     | W m-2      | [365, 600, 1440]       |
| **Wind Speed**        | Daily-Mean Near-Surface Wind Speed         | `sfcWind`  | m s-1      | [365, 600, 1440]       |
| **Max Temperature**   | Daily Maximum Near-Surface Air Temperature | `tasmax`   | K          | [365, 600, 1440]       |
| **Min Temperature**   | Near-Surface Air Temperature (Daily Min)   | `tasmin`   | K          | [365, 600, 1440]       |

**Coordenadas Globais (Amostra 2015):**
*   **Latitude:** -59.88° a 89.88° (Tamanho: 600)
*   **Longitude:** 0.12° a 359.88° (Tamanho: 1440)

histórico

In [ ]:
import os
import shutil
import glob
import gc
import re
import numpy as np
import dask
from dask.diagnostics import ProgressBar
import xarray as xr
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import geopandas as gpd
import rioxarray

# ==========================================
# 1. DEFINIÇÃO DE PARÂMETROS E DIRETÓRIOS
# ==========================================
dask.config.set(scheduler='single-threaded')

MODELOS = ['GFDL-ESM4','IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0','UKESM1-0-LL']
CENARIOS = ['historical']

# 👇 ALTERE AQUI PARA CADA ABA (Exemplo: Aba 1)
VARIAVEIS = ['hurs', 'pr']
# Aba 2: ['rsds', 'sfcWind']
# Aba 3: ['tasmax', 'tasmin']

AWS_BUCKET_NAME = 'nex-gddp-cmip6'
BASE_PREFIX = 'NEX-GDDP-CMIP6'
ANO_INICIO = 1950
ANO_FIM = 2014

DIRETORIO_SAIDA_BASE = '/content/drive/Shareddrives/GAS-Henrique/NEX-GDDP-CMIP6'
TEMP_NC = '/content/temp_nc'

s3_config = Config(
    signature_version=UNSIGNED,
    retries={'max_attempts': 5, 'mode': 'standard'},
    connect_timeout=10,
    read_timeout=30
)
s3 = boto3.client('s3', config=s3_config)
paginator = s3.get_paginator('list_objects_v2')

# ==========================================
# 2. FUNÇÕES DE PROCESSAMENTO
# ==========================================
def limpar_temp():
    if os.path.exists(TEMP_NC):
        shutil.rmtree(TEMP_NC, ignore_errors=True)
    os.makedirs(TEMP_NC, exist_ok=True)

def padronizar_dataset(ds):
    if 'pr' in ds:
        ds['pr'] = ds['pr'] * 86400
        ds['pr'].attrs['units'] = 'mm/day'
    if 'rsds' in ds:
        ds['rsds'] = ds['rsds'] * 0.0864
        ds['rsds'].attrs['units'] = 'MJ m-2 day-1'
    if 'tasmax' in ds:
        ds['tasmax'] = ds['tasmax'] - 273.15
        ds['tasmax'].attrs['units'] = 'degC'
    if 'tasmin' in ds:
        ds['tasmin'] = ds['tasmin'] - 273.15
        ds['tasmin'].attrs['units'] = 'degC'

    if 'lon' in ds.coords:
        ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
        ds = ds.sortby(ds.lon)
    if 'lat' in ds.coords:
        ds = ds.sortby(ds.lat)
    return ds

# --- MAPEAMENTO LOCAL PRÉVIO ---
print("Mapeando arquivos já existentes no Google Drive em memória...")
arquivos_processados = set()
if os.path.exists(DIRETORIO_SAIDA_BASE):
    for root, _, files in os.walk(DIRETORIO_SAIDA_BASE):
        for f in files:
            if f.endswith('.nc4'):
                arquivos_processados.add(f)
print(f"✅ {len(arquivos_processados)} arquivos ignorados por já estarem concluídos.\n")

# Carrega o shapefile
CAMINHO_SHAPEFILE = '/content/drive/Shareddrives/GAS-Henrique/shapefiles.shp'
print("Carregando limites continentais...")
gdf_continentes = gpd.read_file(CAMINHO_SHAPEFILE)


# ==========================================
# 3. LOOP PRINCIPAL
# ==========================================
try:
    for modelo in MODELOS:
        for cenario in CENARIOS:
            print(f"\n[{modelo} | {cenario}] Iniciando mapeamento no S3...")
            prefixo_busca = f"{BASE_PREFIX}/{modelo}/{cenario}/"

            try:
                for page in paginator.paginate(Bucket=AWS_BUCKET_NAME, Prefix=prefixo_busca):
                    if 'Contents' not in page: continue

                    for obj in page['Contents']:
                        key = obj['Key']
                        if not key.endswith('.nc'): continue

                        nome_arq = key.split('/')[-1]
                        match_var = re.match(r'^([a-zA-Z0-9]+)_', nome_arq)
                        match_ano = re.search(r'_(\d{4})(?:_v2\.0)?\.nc$', nome_arq)

                        if not match_var or not match_ano: continue

                        var_arq = match_var.group(1)
                        ano_arq = int(match_ano.group(1))

                        if (var_arq in VARIAVEIS) and (ANO_INICIO <= ano_arq <= ANO_FIM):

                            nome_saida = f"{modelo}_{cenario}_{var_arq}_{ano_arq}.nc4"

                            # Checkpoint local
                            if nome_saida in arquivos_processados:
                                continue

                            pasta_saida = os.path.join(DIRETORIO_SAIDA_BASE, modelo, cenario, var_arq)
                            os.makedirs(pasta_saida, exist_ok=True)
                            arquivo_saida_drive = os.path.join(pasta_saida, nome_saida)

                            print(f"  -> Processando: {nome_saida}")
                            limpar_temp()
                            caminho_local_in = os.path.join(TEMP_NC, nome_arq)
                            caminho_local_out = os.path.join('/content', nome_saida)

                            try:
                                # 1. Download
                                s3.download_file(AWS_BUCKET_NAME, key, caminho_local_in)

                                # 2. Abertura e Padronização (SEM conversão forçada de calendário)
                                time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
                                ds = xr.open_dataset(caminho_local_in, decode_times=time_coder)
                                ds = padronizar_dataset(ds)

                                # 3. Recorte Espacial
                                ds.rio.write_crs("epsg:4326", inplace=True)
                                ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
                                ds = ds.rio.clip(gdf_continentes.geometry, gdf_continentes.crs, drop=True)

                                # 4. Compressão
                                if var_arq in ds.data_vars:
                                    ds[var_arq].encoding = {
                                        'zlib': True,
                                        'complevel': 5,
                                        '_FillValue': np.nan
                                    }

                                # 5. Salva local e move para o Drive
                                ds.to_netcdf(caminho_local_out, engine='h5netcdf', format='NETCDF4')
                                ds.close()

                                shutil.move(caminho_local_out, arquivo_saida_drive)
                                arquivos_processados.add(nome_saida)

                            except Exception as e:
                                print(f"     ❌ Erro em {nome_saida}: {e}")
                                if os.path.exists(caminho_local_out): os.remove(caminho_local_out)
                            finally:
                                if 'ds' in locals(): del ds
                                gc.collect()

            except Exception as e:
                print(f"    ❌ Falha de rede ao listar S3 ({modelo}/{cenario}): {e}")
                continue

except KeyboardInterrupt:
    print("\n⛔ Execução interrompida manualmente pelo usuário.")

limpar_temp()
print("\nProcessamento atomizado finalizado.")

cenários

In [ ]:
import os
import shutil
import glob
import gc
import re
import numpy as np
import dask
from dask.diagnostics import ProgressBar
import xarray as xr
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import geopandas as gpd
import rioxarray

# 1. DEFINIÇÃO DE PARÂMETROS E DIRETÓRIOS
dask.config.set(scheduler='single-threaded')

MODELOS = ['GFDL-ESM4','IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0','UKESM1-0-LL']
CENARIOS = ['ssp126', 'ssp245', 'ssp585']
VARIAVEIS = ['hurs', 'pr', 'rsds', 'sfcWind', 'tasmax', 'tasmin']

AWS_BUCKET_NAME = 'nex-gddp-cmip6'
BASE_PREFIX = 'NEX-GDDP-CMIP6'
ANO_INICIO = 2015
ANO_FIM = 2100

DIRETORIO_SAIDA_BASE = '/content/drive/Shareddrives/GAS-Henrique/NEX-GDDP-CMIP6'
TEMP_NC = '/content/temp_nc'

s3_config = Config(
    signature_version=UNSIGNED,
    retries={'max_attempts': 5, 'mode': 'standard'},
    connect_timeout=10,
    read_timeout=30
)
s3 = boto3.client('s3', config=s3_config)
paginator = s3.get_paginator('list_objects_v2')

# 2. FUNÇÕES DE PROCESSAMENTO
def limpar_temp():
    if os.path.exists(TEMP_NC):
        shutil.rmtree(TEMP_NC, ignore_errors=True)
    os.makedirs(TEMP_NC, exist_ok=True)

def padronizar_dataset(ds):
    if 'pr' in ds:
        ds['pr'] = ds['pr'] * 86400
        ds['pr'].attrs['units'] = 'mm/day'
    if 'rsds' in ds:
        ds['rsds'] = ds['rsds'] * 0.0864
        ds['rsds'].attrs['units'] = 'MJ m-2 day-1'
    if 'tasmax' in ds:
        ds['tasmax'] = ds['tasmax'] - 273.15
        ds['tasmax'].attrs['units'] = 'degC'
    if 'tasmin' in ds:
        ds['tasmin'] = ds['tasmin'] - 273.15
        ds['tasmin'].attrs['units'] = 'degC'

    if 'lon' in ds.coords:
        ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
        ds = ds.sortby(ds.lon)
    if 'lat' in ds.coords:
        ds = ds.sortby(ds.lat)
    return ds

# --- MAPEAMENTO LOCAL PRÉVIO ---
print("Mapeando arquivos já existentes no Google Drive em memória...")
arquivos_processados = set()
if os.path.exists(DIRETORIO_SAIDA_BASE):
    for root, _, files in os.walk(DIRETORIO_SAIDA_BASE):
        for f in files:
            if f.endswith('.nc4'):
                arquivos_processados.add(f)
print(f"✅ {len(arquivos_processados)} arquivos ignorados por já estarem concluídos.\n")

# Carrega o shapefile atualizado
CAMINHO_SHAPEFILE = '/content/drive/Shareddrives/GAS-Henrique/shapefiles.shp'
print("Carregando limites continentais...")
gdf_continentes = gpd.read_file(CAMINHO_SHAPEFILE)

# 3. LOOP PRINCIPAL
try:
    for modelo in MODELOS:
        for cenario in CENARIOS:
            print(f"\n[{modelo} | {cenario}] Iniciando mapeamento no S3...")
            prefixo_busca = f"{BASE_PREFIX}/{modelo}/{cenario}/"

            try:
                for page in paginator.paginate(Bucket=AWS_BUCKET_NAME, Prefix=prefixo_busca):
                    if 'Contents' not in page: continue

                    for obj in page['Contents']:
                        key = obj['Key']
                        if not key.endswith('.nc'): continue

                        nome_arq = key.split('/')[-1]
                        match_var = re.match(r'^([a-zA-Z0-9]+)_', nome_arq)
                        match_ano = re.search(r'_(\d{4})(?:_v2\.0)?\.nc$', nome_arq)

                        if not match_var or not match_ano:
                            continue

                        var_arq = match_var.group(1)
                        ano_arq = int(match_ano.group(1))

                        if (var_arq in VARIAVEIS) and (ANO_INICIO <= ano_arq <= ANO_FIM):

                            nome_saida = f"{modelo}_{cenario}_{var_arq}_{ano_arq}.nc4"

                            # Checkpoint local ultra-rápido
                            if nome_saida in arquivos_processados:
                                continue

                            # Criação da estrutura de pastas sob demanda
                            pasta_saida = os.path.join(DIRETORIO_SAIDA_BASE, modelo, cenario, var_arq)
                            os.makedirs(pasta_saida, exist_ok=True)
                            arquivo_saida_drive = os.path.join(pasta_saida, nome_saida)

                            print(f"  -> Processando: {nome_saida}")
                            limpar_temp()
                            caminho_local_in = os.path.join(TEMP_NC, nome_arq)
                            caminho_local_out = os.path.join('/content', nome_saida)

                            try:
                                # 1. Download
                                s3.download_file(AWS_BUCKET_NAME, key, caminho_local_in)

                                # 2. Abertura e Padronização
                                ds = xr.open_dataset(caminho_local_in)
                                ds = ds.convert_calendar("noleap")
                                ds = padronizar_dataset(ds)

                                # 3. Recorte Espacial
                                ds.rio.write_crs("epsg:4326", inplace=True)
                                ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
                                ds = ds.rio.clip(gdf_continentes.geometry, gdf_continentes.crs, drop=True)

                                # --- NOVO: Força a compressão padrão para a variável ativa ---
                                if var_arq in ds.data_vars:
                                    ds[var_arq].encoding = {
                                        'zlib': True,
                                        'complevel': 5,
                                        '_FillValue': np.nan
                                    }
                                # -------------------------------------------------------------

                                # 4. Salva localmente (VM)
                                ds.to_netcdf(caminho_local_out, engine='h5netcdf', format='NETCDF4')
                                ds.close()

                                # 5. Move com segurança para o Drive e atualiza cache
                                shutil.move(caminho_local_out, arquivo_saida_drive)
                                arquivos_processados.add(nome_saida)

                            except Exception as e:
                                print(f"     ❌ Erro em {nome_saida}: {e}")
                                if os.path.exists(caminho_local_out): os.remove(caminho_local_out)
                            finally:
                                if 'ds' in locals(): del ds
                                gc.collect()

            except Exception as e:
                print(f"    ❌ Falha de rede ao listar S3 ({modelo}/{cenario}): {e}")
                continue

except KeyboardInterrupt:
    print("\n⛔ Execução interrompida manualmente pelo usuário.")

limpar_temp()
print("\nProcessamento atomizado finalizado.")

# Juntando arquivos processados em um único arquivo .nc4
ERA5

In [ ]:
import xarray as xr
import glob
from dask.diagnostics import ProgressBar

# 1. DEFINIR OS CAMINHOS LOCAIS
pasta_in = '/home/henrique/GAS2-Henrique/ERA5'
padrao = f"{pasta_in}/e_*.nc"

# Vamos salvar direto no destino.
arquivo_destino_final = f"{pasta_in}/ERA5_merged.nc4"

ficheiros = sorted(glob.glob(padrao))
print(f"🌍 Encontrados {len(ficheiros)} ficheiros do ERA5 para fundir.")

# 2. ESCUDO DE LIMPEZA
def limpar_era5(ds):
    if 'latitude' in ds.coords: ds = ds.rename({'latitude': 'lat'})
    if 'longitude' in ds.coords: ds = ds.rename({'longitude': 'lon'})

    if 'height' in ds:
        ds = ds.drop_vars('height')

    ds = ds.sortby('lat').sortby('lon')

    # Mantendo como float32 (se o original for) para poupar muito espaço no SSD
    ds.coords['lat'] = np.round(ds.coords['lat'].values, 3)
    ds.coords['lon'] = np.round(ds.coords['lon'].values, 3)
    return ds

if not ficheiros:
    print("❌ Nenhum ficheiro encontrado! Verifique o caminho da pasta.")
else:
    print("⏳ A carregar o Hipercubo (via Dask)...")

    # Como você tem muita RAM, chunks de 360 ou até maiores vão voar no processamento
    ds_mega = xr.open_mfdataset(
        ficheiros,
        combine='by_coords',
        chunks={'time': 360},
        preprocess=limpar_era5
    )

    # Nível 4 de compressão é o ideal entre tamanho e velocidade
    compressao = {var: {'zlib': True, 'complevel': 4} for var in ds_mega.data_vars}

    print(f"💾 A gravar o Hipercubo direto no SSD NVMe...")
    print(f"Destino: {arquivo_destino_final}\n")

    # BARRA DE PROGRESSO DO DASK
    with ProgressBar():
        ds_mega.to_netcdf(arquivo_destino_final, engine='netcdf4', format='NETCDF4', encoding=compressao)

    ds_mega.close()

    print("\n🎉 PROCESSAMENTO CONCLUÍDO! O Monstro está domado e salvo!")

In [ ]:
import xarray as xr
ds = xr.open_dataset('/home/henrique/GAS2-Henrique/ERA5/ERA5_merged.nc4', chunks='auto')
ds

CMIP6

In [ ]:
import xarray as xr
import glob
import numpy as np
import os
import shutil
from dask.diagnostics import ProgressBar

# aumentar o cache de arquivos para evitar que o NetCDF feche arquivos a meio!
xr.set_options(file_cache_maxsize=1200)

# 1. DEFINIÇÕES GERAIS
pasta_in = '/home/henrique/GAS2-Henrique/CMIP6'
pasta_tmp = '/home/henrique/Temp_Memmaps'
os.makedirs(pasta_tmp, exist_ok=True)

gcms = ['CMCC-ESM2', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-MM']
cenarios = ['ssp126', 'ssp245', 'ssp585']

# 2. FUNÇÃO DE LIMPEZA
def limpar_cmip6(ds):
    if 'latitude' in ds.coords: ds = ds.rename({'latitude': 'lat'})
    if 'longitude' in ds.coords: ds = ds.rename({'longitude': 'lon'})
    if 'height' in ds: ds = ds.drop_vars('height')

    if 'lat' in ds.coords and 'lon' in ds.coords:
        ds = ds.sortby('lat').sortby('lon')
        ds.coords['lat'] = np.round(ds.coords['lat'].values.astype('float64'), 3)
        ds.coords['lon'] = np.round(ds.coords['lon'].values.astype('float64'), 3)

    # REMOVER VARIÁVEIS DE BOUNDS (Evita falhas na hora de juntar os arquivos)
    vars_to_drop = [v for v in ds.variables if 'bnds' in v or 'bounds' in v]
    if vars_to_drop:
        ds = ds.drop_vars(vars_to_drop)

    return ds

# 3. MOTOR DE PROCESSAMENTO
print("🚀 Iniciando a máquina de fusão do CMIP6...\n")

for gcm in gcms:
    # ---------------------------------------------------------
    # A) PROCESSAR O HISTÓRICO
    # ---------------------------------------------------------
    print(f"--- 📚 Processando HISTÓRICO: {gcm} ---")

    padrao_hist = f"{pasta_in}/c_*_{gcm}_[12][0-9][0-9][0-9].nc"
    arquivos_hist = sorted(glob.glob(padrao_hist))

    if arquivos_hist:
        arquivo_saida_hist = f"{pasta_in}/{gcm}_historico.nc4"
        arquivo_tmp_hist = f"{pasta_tmp}/{gcm}_historico.nc4" # <-- Caminho local temporário

        if not os.path.exists(arquivo_saida_hist):
            print(f"Encontrados {len(arquivos_hist)} arquivos. Fundindo variáveis e anos...")
            ds_hist = xr.open_mfdataset(
                arquivos_hist,
                combine='by_coords',
                chunks={'time': 360},
                preprocess=limpar_cmip6
            )

            # Forçando as coordenadas para float64 (Boa prática que discutimos antes!)
            compressao = {var: {'zlib': True, 'complevel': 4} for var in ds_hist.data_vars}
            compressao['lat'] = {'dtype': 'float64'}
            compressao['lon'] = {'dtype': 'float64'}

            print(f"💾 A gravar localmente em: {arquivo_tmp_hist}")
            with ProgressBar():
                # MUDAR AQUI: engine='h5netcdf'
                ds_hist.to_netcdf(arquivo_tmp_hist, engine='h5netcdf', format='NETCDF4', encoding=compressao)
            ds_hist.close()

            print("🚚 A transferir arquivo do disco local para o Google Drive...")
            shutil.move(arquivo_tmp_hist, arquivo_saida_hist)
            print(f"✅ Salvo no Drive: {arquivo_saida_hist}\n")
        else:
            print(f"⚠️ Arquivo {arquivo_saida_hist} já existe no Drive. Pulando...\n")
    else:
        print(f"❌ Nenhum arquivo histórico encontrado para {gcm}.\n")

    # ---------------------------------------------------------
    # B) PROCESSAR AS PROJEÇÕES
    # ---------------------------------------------------------
    for cenario in cenarios:
        print(f"--- 🔮 Processando PROJEÇÃO: {gcm} | {cenario} ---")

        padrao_proj = f"{pasta_in}/c_*_{gcm}_{cenario}_*.nc"
        arquivos_proj = sorted(glob.glob(padrao_proj))

        if arquivos_proj:
            arquivo_saida_proj = f"{pasta_in}/{gcm}_{cenario}.nc4"
            arquivo_tmp_proj = f"{pasta_tmp}/{gcm}_{cenario}.nc4" # <-- Caminho local temporário

            if not os.path.exists(arquivo_saida_proj):
                print(f"Encontrados {len(arquivos_proj)} arquivos. Fundindo variáveis e anos...")
                ds_proj = xr.open_mfdataset(
                    arquivos_proj,
                    combine='by_coords',
                    chunks={'time': 360},
                    preprocess=limpar_cmip6
                )

                compressao = {var: {'zlib': True, 'complevel': 4} for var in ds_proj.data_vars}
                compressao['lat'] = {'dtype': 'float64'}
                compressao['lon'] = {'dtype': 'float64'}

                print(f"💾 A gravar localmente em: {arquivo_tmp_proj}")
                with ProgressBar():
                    # MUDAR AQUI: engine='h5netcdf'
                    ds_proj.to_netcdf(arquivo_tmp_proj, engine='h5netcdf', format='NETCDF4', encoding=compressao)
                ds_proj.close()

                print("🚚 A transferir arquivo do disco local para o Google Drive...")
                shutil.move(arquivo_tmp_proj, arquivo_saida_proj)
                print(f"✅ Salvo no Drive: {arquivo_saida_proj}\n")
            else:
                print(f"⚠️ Arquivo {arquivo_saida_proj} já existe no Drive. Pulando...\n")
        else:
            print(f"❌ Nenhum arquivo de projeção encontrado para {gcm} - {cenario}.\n")

print("🎉 PROCESSAMENTO FINALIZADO!")

In [ ]:
import xarray as xr
xr.open_dataset('/home/henrique/GAS2-Henrique/CMIP6/MPI-ESM1-2-HR_ssp126.nc4', chunks='auto')

In [ ]:
import glob
import xarray as xr
from tqdm import tqdm

# Aponta para os arquivos exatos onde o script falhou
padrao = '/home/henrique/GAS2-Henrique/CMIP6/c_*_MRI-ESM2-0_ssp245_*.nc'
arquivos = sorted(glob.glob(padrao))

print(f"🔍 A verificar integridade de {len(arquivos)} arquivos...\n")

arquivos_corrompidos = []

for arq in tqdm(arquivos, desc="Scanner HDF5"):
    try:
        # Tenta abrir e ler os metadados do tempo (força a leitura do disco)
        with xr.open_dataset(arq, engine='netcdf4') as ds:
            _ = ds.time.values
    except Exception as e:
        arquivos_corrompidos.append(arq)
        print(f"\n❌ CORROMPIDO: {arq}")
        print(f"   -> Detalhe do Erro: {e}")

print("\n" + "="*60)
if not arquivos_corrompidos:
    print("✅ RESULTADO: Todos os arquivos estão estruturalmente perfeitos!")
    print("💡 Conclusão: O problema não é corrupção. É garantidamente o limite de cache do Xarray. Usa o `xr.set_options(file_cache_maxsize=1200)` ou `engine='h5netcdf'` como conversámos.")
else:
    print(f"🚨 RESULTADO: Encontrados {len(arquivos_corrompidos)} arquivos corrompidos.")
    print("💡 Solução: Apaga estes arquivos específicos e volta a descarregá-los do Copernicus/Earth System Grid.")
print("="*60)